# KORMo 1B — from-scratch 사전학습 (Colab A100 전용)

[MLP-Lab/KORMo-tutorial](https://github.com/MLP-Lab/KORMo-tutorial)의 `01.pretrain_from_scratch.ipynb`를 Colab A100에서 바로 실행되도록 조정한 버전입니다.

**사용법**: 런타임 → 런타임 유형 변경 → **A100 GPU** 선택 → 저장 → 런타임 → 모두 실행

원본 대비 변경점:
- uv 가상환경·flash-attn 빌드 제거 (본 노트북은 PyTorch 내장 `flex_attention` 사용 → flash-attn 불필요)
- `num_proc`을 Colab vCPU 수에 맞게 자동 설정 (원본: 48/128 하드코딩)
- 체크포인트를 Google Drive에 저장하는 옵션 추가 (세션 끊김 대비)

예상 소요: 데이터 준비 ~10분 + 학습 1~2시간 (A100 40GB 기준)

In [ ]:
# GPU 확인 — A100이 보여야 합니다
!nvidia-smi

## 0. 환경 준비

Colab에 torch·accelerate·pyyaml은 이미 있으므로 transformers 버전 고정과 datasets만 설치합니다.
(원본 리포가 고정한 4.57.0은 yank된 버전이라 패치 버전 4.57.1 사용)

In [ ]:
!git clone https://github.com/MLP-Lab/KORMo-tutorial.git /content/KORMo-tutorial
%pip install -q "transformers==4.57.1" "datasets>=4.1.1"

import os, sys
sys.path.append('/content/KORMo-tutorial/src')
NUM_PROC = os.cpu_count()
print(f"vCPU: {NUM_PROC}")

### (선택) Google Drive에 체크포인트 저장

`USE_DRIVE = True`로 바꾸면 세션이 끊겨도 체크포인트가 남습니다.
Drive 무료 용량(15GB)을 고려해 Drive 사용 시 optimizer 상태 없이 모델 가중치만 저장합니다(체크포인트당 약 2.6GB).

In [ ]:
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/kormo-1B-PT'
else:
    OUTPUT_DIR = '/content/kormo-1B-PT'
print("체크포인트 저장 위치:", OUTPUT_DIR)

## 1. 데이터 준비 — Raw text → Tokenize → Sequence packing

토크나이저는 `KORMo-Team/KORMo-tokenizer`(vocab 125,000), 데이터는 튜토리얼용 소형 코퍼스를 사용합니다.
`tokenizer.encode()`가 문서 앞에 `<|BOS|>`를 자동으로 붙이는데, 이 BOS 위치가 뒤에서 문서 경계 마스크의 기준이 됩니다.

In [ ]:
import datasets
from transformers import AutoTokenizer

pt_dataset = datasets.load_dataset(
    'KORMo-Team/KORMo-tutorial-datasets', name='pretrain', split='train'
)
train_ds = pt_dataset.shuffle(seed=42)
print(train_ds)

tokenizer = AutoTokenizer.from_pretrained('KORMo-Team/KORMo-tokenizer')
print("vocab:", tokenizer.vocab_size, "| bos:", tokenizer.bos_token, "| eos:", tokenizer.eos_token)

In [ ]:
# 토크나이즈 (문서 끝에 EOT 추가)
def _tokenize(examples, tokenizer):
    input_ids = [tokenizer.encode(text) + [tokenizer.eos_token_id] for text in examples['text']]
    return {'input_ids': input_ids}

tokenized_ds = train_ds.map(
    _tokenize,
    batched=True,
    num_proc=NUM_PROC,
    remove_columns=train_ds.column_names,
    fn_kwargs={'tokenizer': tokenizer},
)
print(tokenized_ds)

In [ ]:
# 4096 토큰 단위 시퀀스 패킹 — 여러 문서를 이어붙여 패딩 낭비 제거
from itertools import chain

SEQ_LEN = 4096

def _pack_dataset(examples, seq_len):
    flat = list(chain.from_iterable(examples["input_ids"]))
    n_full = len(flat) // seq_len
    return {"input_ids": [flat[i*seq_len:(i+1)*seq_len] for i in range(n_full)]}

packed_ds = tokenized_ds.map(
    _pack_dataset,
    batched=True,
    batch_size=100_000,
    remove_columns=tokenized_ds.column_names,
    num_proc=NUM_PROC,
    fn_kwargs={'seq_len': SEQ_LEN},
)
packed_ds.set_format('torch')
total_tokens = len(packed_ds) * SEQ_LEN
print(packed_ds)
print(f"총 학습 토큰: {total_tokens/1e6:.1f}M")

## 2. Intra-document attention mask (flex_attention)

패킹으로 한 시퀀스에 여러 문서가 섞이므로, BOS 토큰의 누적합으로 문서 ID를 만들어
**같은 문서 안에서만 attention이 가도록** block mask를 만듭니다. causal mask와 AND로 결합합니다.

In [ ]:
import torch
from torch.nn.attention.flex_attention import create_block_mask, and_masks

def _intra_doc_mask(input_ids, bos_token_id):
    is_bos = (input_ids == bos_token_id)
    doc_ids = torch.cumsum(is_bos.flatten().long(), 0).view_as(input_ids)

    def intra_doc(b, h, q_idx, kv_idx):
        return doc_ids[b, q_idx] == doc_ids[b, kv_idx]

    def causal(b, h, q_idx, kv_idx):
        return q_idx >= kv_idx

    return and_masks(intra_doc, causal)

def create_intra_doc_mask(input_ids, tokenizer):
    B, Q_LEN = input_ids.shape
    mask_mod = _intra_doc_mask(input_ids.to('cuda'), tokenizer.bos_token_id)
    return create_block_mask(mask_mod=mask_mod, B=B, H=None, Q_LEN=Q_LEN, KV_LEN=Q_LEN)

In [ ]:
from dataclasses import dataclass
from torch.nn.utils.rnn import pad_sequence
from transformers import PreTrainedTokenizer

@dataclass
class DataCollatorIntraDocMask:
    tokenizer: PreTrainedTokenizer

    def __call__(self, instances):
        input_ids = [inst["input_ids"][:SEQ_LEN] for inst in instances]
        input_ids = pad_sequence(input_ids, batch_first=True,
                                 padding_value=self.tokenizer.pad_token_id)
        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        block_mask = create_intra_doc_mask(input_ids, self.tokenizer)
        return dict(input_ids=input_ids, labels=labels, attention_mask=block_mask)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

## 3. 모델 초기화 + 학습

1B config: hidden 2048 / 16 layers / GQA(16 heads, 8 KV) / vocab 125,184 → 약 1.3B 파라미터, bf16.
A100 40GB에서 batch 4 × 4096 토큰이 여유 있게 들어갑니다.

> wandb로 로그를 보려면 `report_to='none'`을 `'wandb'`로 바꾸고 본인 API 키로 로그인하세요.

In [ ]:
from kormo.train.arguments import KORMoTrainingArguments
from kormo.train.trainer import KORMoTrainer
from kormo.modeling_configs.load_model import load_model_from_config

model, model_config = load_model_from_config('1B', _attn_implementation='flex_attention')
model.to('cuda')
n_params = sum(p.numel() for p in model.parameters())
print(f"파라미터: {n_params/1e9:.2f}B | attn: {model.config._attn_implementation} | dtype: {next(model.parameters()).dtype}")

In [ ]:
training_arguments = KORMoTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    lr_scheduler_type='linear',
    logging_steps=10,
    save_strategy='steps',
    save_steps=500,
    save_total_limit=2,
    save_only_model=USE_DRIVE,   # Drive 사용 시 optimizer 상태 제외(용량 절약)
    report_to='none',
)

trainer = KORMoTrainer(
    model=model,
    args=training_arguments,
    train_dataset=packed_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorIntraDocMask(tokenizer),
)

In [ ]:
trainer.train()

## 4. 저장 + 생성 테스트

소형 코퍼스 1 epoch이므로 출력 품질은 옹알이 수준이 정상입니다.
loss가 초기 ~11(=ln 125,184, 균등분포)에서 꾸준히 내려왔다면 파이프라인은 정상 동작한 것입니다.

In [ ]:
trainer.save_model(f"{OUTPUT_DIR}/final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final")

model.eval()
prompt = "한국의 수도는"
inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=64, do_sample=True, top_p=0.9)
print(tokenizer.decode(out[0], skip_special_tokens=True))